# What this notebook is, and what it found

**Purpose:** `05_kcc_embedding_indexing.ipynb` showed MuRIL and LaBSE
*separate topics* to different degrees in isolated pair tests. This
notebook answers the more practical question: when a **real, held-out
farmer query** (not seen during indexing) comes in, does the retrieved
result actually share the query's true crop/category \u2014 and does that
beat just picking random chunks?

**Method:** ~200 held-out KCC queries (excluded from the index, stratified
by Category), searched against MuRIL's and LaBSE's indices plus a random
baseline, checking whether the top-5 results share the query's true
Category / Crop.

## Results (this run)

| Method | Category-match@5 | Crop-match@5 | Crop-match lift over random |
|---|---|---|---|
| Random baseline | 61.0% | 26.5% | \u2014 |
| MuRIL | 67.0% | 33.0% | **+6.5%** |
| LaBSE | 69.0% | 42.0% | **+15.5%** |

**Bottom line:** crop-match is the more honest number here (finer-grained,
harder to hit by luck than the broad Category label). MuRIL barely beats
random guessing \u2014 consistent with the near-zero Cohen's d found in
`05`. LaBSE more than doubles MuRIL's lift over random, a real if modest
improvement. Neither is a strong result on its own.

**What happened next:** a third model, `BAAI/bge-m3`, was quick-tested
afterward on the same KCC data (`05b_kcc_bge_m3_quick_test.ipynb`) and
scored **81% crop-match@5 against a 19% random baseline \u2014 a +62% lift**,
roughly 4x LaBSE's improvement and 10x MuRIL's. This notebook's MuRIL/LaBSE
numbers are kept as the baseline that motivated testing bge-m3 in the
first place, and as independent corroboration of the same MuRIL failure
the team's production RAG build found separately (Milestone 3 report,
\u00a75.2, \u00a79.10).


# KCC Retrieval Pipeline \u2014 v2: MuRIL vs LaBSE vs Random Baseline

**Why v2:** v1's 67% category-match@5 for MuRIL had no baseline to compare
against, and the anisotropy diagnostic in `05` v2 showed MuRIL's
same-category / different-category similarity distributions barely
separate (near-uniform 0.98+ regardless of topic). This notebook settles
the question with numbers instead of judgment calls:

1. **Random-retrieval baseline** \u2014 what category-match@k would you get by
   picking 5 random chunks, ignoring the query entirely? If MuRIL barely
   beats this, MuRIL's embedding isn't doing useful work.
2. **MuRIL vs LaBSE**, same eval, same held-out queries, side by side.
3. Same qualitative farmer-style queries as v1, run through both models,
   so you can eyeball the difference directly (not just trust a metric).

**Input:** `kcc_faiss_index_muril.bin`, `kcc_faiss_index_labse.bin`,
`kcc_index_metadata.jsonl` (all from `05_kcc_embedding_indexing.ipynb` v2),
plus `kcc_cleaned_all_crops.csv` for held-out queries.


In [1]:
# Step 0: Install dependencies (uncomment on a fresh Colab runtime)
!pip install -q sentence-transformers faiss-cpu transformers torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 39.9 MB/s eta 0:00:00


In [2]:
# Code using mounted Google Drive (commented out for colleague's convenience; uncomment to run on Colab)
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
import faiss

import warnings
warnings.filterwarnings('ignore')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


Using device: cpu


## Step 1: Load Both Indices + Both Models

In [4]:
BASE_PATH = "/content/drive/MyDrive/kcc_raw/"
PROCESSED_PATH = f"{BASE_PATH}processed/"
FINAL_PATH = f"{BASE_PATH}final/"

paths = {
    "muril_index": f"{FINAL_PATH}kcc_faiss_index_muril.bin",
    "labse_index": f"{FINAL_PATH}kcc_faiss_index_labse.bin",
    "metadata": f"{FINAL_PATH}kcc_index_metadata.jsonl",
    "full_csv": f"{PROCESSED_PATH}kcc_cleaned_all_crops.csv",
}
for name, p in paths.items():
    if not Path(p).exists():
        raise FileNotFoundError(f"\u274c '{p}' not found. Run 05_kcc_embedding_indexing.ipynb (v2) first.")

muril_index = faiss.read_index(paths["muril_index"])
labse_index = faiss.read_index(paths["labse_index"])
print(f"\u2705 MuRIL index: {muril_index.ntotal:,} vectors, dim={muril_index.d}")
print(f"\u2705 LaBSE index: {labse_index.ntotal:,} vectors, dim={labse_index.d}")

indexed_records = []
with open(paths["metadata"], 'r', encoding='utf-8') as f:
    for line in f:
        indexed_records.append(json.loads(line))
print(f"\u2705 Loaded {len(indexed_records):,} metadata records (shared order for both indices)")


✅ MuRIL index: 5,000 vectors, dim=768
✅ LaBSE index: 5,000 vectors, dim=768
✅ Loaded 5,000 metadata records (shared order for both indices)


In [5]:
MURIL_NAME = "google/muril-base-cased"
muril_tokenizer = AutoTokenizer.from_pretrained(MURIL_NAME)
muril_model = AutoModel.from_pretrained(MURIL_NAME).to(DEVICE)
muril_model.eval()

LABSE_NAME = "sentence-transformers/LaBSE"
labse_model = SentenceTransformer(LABSE_NAME, device=DEVICE)

print(f"\u2705 Both models loaded (MuRIL dim={muril_model.config.hidden_size}, "
      f"LaBSE dim={labse_model.get_sentence_embedding_dimension()})")


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def embed_muril_query(text, max_length=512):
    enc = muril_tokenizer([text], padding=True, truncation=True,
                           max_length=max_length, return_tensors="pt").to(DEVICE)
    out = muril_model(**enc)
    pooled = mean_pooling(out.last_hidden_state, enc["attention_mask"])
    pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
    return pooled.cpu().numpy().astype('float32')


def embed_labse_query(text):
    return labse_model.encode([text], normalize_embeddings=True, show_progress_bar=False).astype('float32')


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 2.36MB            

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

✅ Both models loaded (MuRIL dim=768, LaBSE dim=768)


## Step 2: Unified Search Interface (per model) + Random Baseline

In [6]:
def search(query_text, model, k=5):
    """model: 'muril' or 'labse' or 'random'."""
    t0 = time.time()

    if model == "random":
        rng = np.random.default_rng(hash(query_text) % (2**32))
        ids = rng.choice(len(indexed_records), size=k, replace=False)
        scores = np.zeros(k)  # not meaningful for random
    elif model == "muril":
        q_vec = embed_muril_query(query_text)
        scores, ids = muril_index.search(q_vec, k)
        scores, ids = scores[0], ids[0]
    elif model == "labse":
        q_vec = embed_labse_query(query_text)
        scores, ids = labse_index.search(q_vec, k)
        scores, ids = scores[0], ids[0]
    else:
        raise ValueError(f"unknown model {model}")

    latency_ms = (time.time() - t0) * 1000
    results = []
    for score, idx in zip(scores, ids):
        if idx == -1:
            continue
        rec = indexed_records[idx]
        results.append({"id": int(idx), "score": float(score),
                         "text": rec["text"], "metadata": rec["metadata"]})
    return results, latency_ms


for m in ["muril", "labse", "random"]:
    res, lat = search("wheat crop disease treatment", model=m, k=3)
    print(f"{m:6s} wired correctly \u2014 {len(res)} results in {lat:.1f}ms")


muril  wired correctly — 3 results in 282.0ms
labse  wired correctly — 3 results in 162.7ms
random wired correctly — 3 results in 0.3ms


## Step 3: Self-Retrieval Recall@k (all three, for completeness)

Random retrieval is included here too \u2014 it should score ~0% self-retrieval
(picking your own exact chunk out of 5,000 by chance at k=10 is ~0.2%),
which is a useful gut-check that this metric alone doesn't distinguish
good embeddings from bad ones. The real test is Step 4.

In [7]:
def recall_at_k(hit_ranks, k_values):
    out = {}
    for k in k_values:
        hits = sum(1 for r in hit_ranks if r is not None and r <= k)
        out[k] = hits / len(hit_ranks)
    return out


def mrr(hit_ranks):
    return sum(1.0 / r if r is not None else 0.0 for r in hit_ranks) / len(hit_ranks)


K_VALUES = [1, 3, 5, 10]
MAX_K = max(K_VALUES)

rng = np.random.default_rng(42)
N_SELF = 150
self_sample_ids = rng.choice(len(indexed_records), size=min(N_SELF, len(indexed_records)), replace=False)

self_results = {}
for model in ["muril", "labse", "random"]:
    ranks = []
    for idx in self_sample_ids:
        query_text = indexed_records[idx]["text"]
        results, _ = search(query_text, model=model, k=MAX_K)
        rank = next((pos for pos, r in enumerate(results, 1) if r["id"] == idx), None)
        ranks.append(rank)
    self_results[model] = {"recall": recall_at_k(ranks, K_VALUES), "mrr": mrr(ranks)}

print(f"{'Model':8s} " + " ".join(f"R@{k:<4d}" for k in K_VALUES) + "   MRR")
for model, res in self_results.items():
    row = " ".join(f"{res['recall'][k]:5.1%}" for k in K_VALUES)
    print(f"{model:8s} {row}   {res['mrr']:.3f}")


Model    R@1    R@3    R@5    R@10     MRR
muril    100.0% 100.0% 100.0% 100.0%   1.000
labse    100.0% 100.0% 100.0% 100.0%   1.000
random    0.0%  0.0%  0.0%  0.0%   0.000


## Step 4: Held-Out Generalization \u2014 MuRIL vs LaBSE vs Random Baseline

This is the number that actually matters. Same held-out query
construction as v1 (queries excluded from the index, stratified by
Category), run through all three retrieval methods so LaBSE's and
MuRIL's category/crop-match rates can be read against a real floor,
not an assumed one.

In [8]:
full_df = pd.read_csv(paths["full_csv"])
print(f"Full cleaned dataset: {len(full_df):,} rows")

indexed_prefixes = set(r["text"][:80] for r in indexed_records)

def qa_prefix(row):
    q = str(row.get('cleaned_query', row.get('QueryText', '')))
    a = str(row.get('cleaned_answer', row.get('KccAns', '')))
    return f"Question: {q}\nAnswer: {a}"[:80]

full_df['_prefix'] = full_df.apply(qa_prefix, axis=1)
held_out_df = full_df[~full_df['_prefix'].isin(indexed_prefixes)].copy()
print(f"Held-out (not in index) candidates: {len(held_out_df):,}")


def stratified_df_sample(df, n_total, strat_col='Category', random_state=42):
    frames = []
    for cat, group in df.groupby(strat_col):
        share = len(group) / len(df)
        n_take = max(1, round(share * n_total))
        n_take = min(n_take, len(group))
        frames.append(group.sample(n_take, random_state=random_state))
    out = pd.concat(frames)
    if len(out) > n_total:
        out = out.sample(n_total, random_state=random_state)
    return out.reset_index(drop=True)

N_EVAL = 200
eval_df = stratified_df_sample(held_out_df, N_EVAL, strat_col='Category', random_state=42)
query_col = 'cleaned_query' if 'cleaned_query' in eval_df.columns else 'QueryText'
print(f"Held-out eval set: {len(eval_df):,} queries")


Full cleaned dataset: 710,616 rows
Held-out (not in index) candidates: 674,178
Held-out eval set: 200 queries


In [9]:
K_FOR_MATCH = 5
comparison = {}

for model in ["muril", "labse", "random"]:
    category_hits, crop_hits, latencies = [], [], []
    for _, row in eval_df.iterrows():
        query_text = str(row[query_col])
        if not query_text.strip():
            continue
        results, latency_ms = search(query_text, model=model, k=K_FOR_MATCH)
        latencies.append(latency_ms)
        true_category = row.get('Category', 'unknown')
        true_crop = row.get('Crop', 'unknown')
        category_hits.append(any(r['metadata'].get('category') == true_category for r in results))
        crop_hits.append(any(r['metadata'].get('crop') == true_crop for r in results))

    comparison[model] = {
        "n": len(category_hits),
        "category_match_rate": float(np.mean(category_hits)),
        "crop_match_rate": float(np.mean(crop_hits)),
        "latency_ms_mean": float(np.mean(latencies)),
        "latency_ms_p50": float(np.percentile(latencies, 50)),
    }

print(f"Held-Out Generalization Evaluation (n={comparison['muril']['n']}, k={K_FOR_MATCH})")
print("-" * 70)
print(f"{'Model':8s} {'Category@5':>12s} {'Crop@5':>10s} {'Latency (mean/p50 ms)':>24s}")
for model, res in comparison.items():
    print(f"{model:8s} {res['category_match_rate']:11.1%} {res['crop_match_rate']:9.1%} "
          f"{res['latency_ms_mean']:10.1f} / {res['latency_ms_p50']:.1f}")

print(f"\nLift over random baseline:")
print(f"  MuRIL: {comparison['muril']['category_match_rate'] - comparison['random']['category_match_rate']:+.1%}")
print(f"  LaBSE: {comparison['labse']['category_match_rate'] - comparison['random']['category_match_rate']:+.1%}")


Held-Out Generalization Evaluation (n=200, k=5)
----------------------------------------------------------------------
Model      Category@5     Crop@5    Latency (mean/p50 ms)
muril          67.0%     33.0%      107.2 / 98.9
labse          69.0%     42.0%      128.3 / 117.0
random         61.0%     26.5%        0.1 / 0.1

Lift over random baseline:
  MuRIL: +6.0%
  LaBSE: +8.0%


## Step 5: Qualitative Comparison \u2014 Same Queries, MuRIL vs LaBSE

Same 10 farmer-style queries as v1 (English / Hinglish / Hindi mix), now
run through both models side by side so you can eyeball whether LaBSE's
top-1 result is actually more relevant, not just numerically different.

In [10]:
TEST_QUERIES = [
    "wheat crop is turning yellow what to do",
    "gehu mein pila rog laga hai kya kare",
    "\u0917\u0947\u0939\u0942\u0902 \u0915\u0940 \u092b\u0938\u0932 \u092e\u0947\u0902 \u092a\u0940\u0932\u093e \u0930\u094b\u0917 \u0915\u094d\u092f\u093e \u0915\u0930\u0947\u0902",
    "best fertilizer for rice paddy",
    "PM Kisan yojana eligibility kaise check kare",
    "paddy pest control organic method",
    "mandi bhav for wheat today",
    "sugarcane disease red rot treatment",
    "how much water needed for maize crop",
    "onion price today up",
]

for q in TEST_QUERIES:
    print(f"\nQuery: {q}")
    for model in ["muril", "labse"]:
        results, latency_ms = search(q, model=model, k=2)
        print(f"  [{model}] ({latency_ms:.0f}ms)")
        for rank, r in enumerate(results, start=1):
            print(f"    [{rank}] score={r['score']:.3f} | crop={r['metadata'].get('crop')} | "
                  f"category={r['metadata'].get('category')}")
            print(f"         {r['text'][:120].replace(chr(10), ' ')}...")



Query: wheat crop is turning yellow what to do
  [muril] (98ms)
    [1] score=0.995 | crop=Wheat | category=Cereals
         Question: Please give me information nilgai are harming the whaet crop. ?...
    [2] score=0.995 | crop=Tomato | category=Vegetables
         Question: Information about Leaves are bent in tomato crop and black and white spots are also visible in it. Growth and ...
  [labse] (97ms)
    [1] score=0.450 | crop=Paddy (Dhan) | category=Cereals
         Question: Paddy nursery is turning yellow, what to do ? Answer: श्रीमान जी, धान की नर्सरी में 2 ग्राम ज़िंक सल्फेट 33 , ...
    [2] score=0.427 | crop=Wheat | category=Cereals
         Question: Wheat crop has fallen...? Answer: सर अगर गेहूं की फसल गिर गई है तो नुकसान होने की संभावना बनी रहती है...

Query: gehu mein pila rog laga hai kya kare
  [muril] (105ms)
    [1] score=0.993 | crop=Paddy (Dhan) | category=Cereals
         Question: dhan me vikash kam ho raha hai ? Answer: srimaan ji aap dhan ki fasal me urea saga

## Step 6: Verdict + Save Comparison Summary

Decision rule for the report: if LaBSE's category-match@5 lift over
random is meaningfully larger than MuRIL's (and Step 4 in `05` v2 showed
a materially higher Cohen's d), that's the evidence to formally switch
the architecture doc's embedding model \u2014 not a subjective call.

In [11]:
verdict = "labse" if comparison["labse"]["category_match_rate"] > comparison["muril"]["category_match_rate"] else "muril"

summary = {
    "held_out_eval": comparison,
    "self_retrieval": {m: {"recall": r["recall"], "mrr": r["mrr"]} for m, r in self_results.items()},
    "recommended_model": verdict,
    "known_limitations": [
        "Category/crop-match is a relevance proxy, not human judgment.",
        "Index built on a 5,000-chunk stratified sample, not the full corpus.",
        "Random baseline uses a fixed-seed-per-query pseudo-random draw for reproducibility, not true uniform random each run.",
        "Recommend a manually-labeled 100-150 pair relevance set before Milestone 5 regardless of which model is chosen.",
    ],
}

with open(f"{FINAL_PATH}kcc_retrieval_eval_comparison_summary.json", 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"Recommended embedding model based on this run: {verdict.upper()}")
print(f"\nSaved: {FINAL_PATH}kcc_retrieval_eval_comparison_summary.json")
print(json.dumps(summary, indent=2, ensure_ascii=False))


Recommended embedding model based on this run: LABSE

Saved: /content/drive/MyDrive/kcc_raw/final/kcc_retrieval_eval_comparison_summary.json
{
  "held_out_eval": {
    "muril": {
      "n": 200,
      "category_match_rate": 0.67,
      "crop_match_rate": 0.33,
      "latency_ms_mean": 107.19943523406982,
      "latency_ms_p50": 98.9142656326294
    },
    "labse": {
      "n": 200,
      "category_match_rate": 0.69,
      "crop_match_rate": 0.42,
      "latency_ms_mean": 128.25030088424683,
      "latency_ms_p50": 117.00630187988281
    },
    "random": {
      "n": 200,
      "category_match_rate": 0.61,
      "crop_match_rate": 0.265,
      "latency_ms_mean": 0.07943987846374512,
      "latency_ms_p50": 0.07259845733642578
    }
  },
  "self_retrieval": {
    "muril": {
      "recall": {
        "1": 1.0,
        "3": 1.0,
        "5": 1.0,
        "10": 1.0
      },
      "mrr": 1.0
    },
    "labse": {
      "recall": {
        "1": 1.0,
        "3": 1.0,
        "5": 1.0,
       

---
## Next Steps

1. Update the Milestone 3 architecture doc's embedding model choice based
   on the verdict above, with the Cohen's d and category-match numbers as
   the justification \u2014 this turns "we chose MuRIL" into a measured
   decision rather than an assumption.
2. If LaBSE wins here but you need a **fine-tuned domain-specific** model
   for Milestone 4/5 (better still than an off-the-shelf general-purpose
   sentence embedder), consider a SimCSE-style contrastive fine-tune of
   LaBSE or MuRIL on your own Q\u2013A pairs as a stretch goal.
3. Get a small manually-labeled relevance set going (100\u2013150 pairs) before
   Milestone 5 \u2014 flagged in every version of this notebook so it doesn't
   keep getting deferred.
